# Mini Projet Préparation de Données Images
Classification de déchets : cardboard, glass, metal, paper, plastic, trash

Objectif : construire un jeu de données images propre et homogène à partir d'images brutes hétérogènes.

## Configuration et structure du projet

In [1]:
# Question 0 - Import des librairies
import os
import shutil
import hashlib
import numpy as np
import pandas as pd
from PIL import Image, ImageStat
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [6]:
# Question 0 - Définition des chemins et création des dossiers du projet
BASE_DIR = ".."
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
CLEANED_DIR = os.path.join(BASE_DIR, "data", "cleaned")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")

CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]

for classe in CLASSES:
    os.makedirs(os.path.join(CLEANED_DIR, classe), exist_ok=True)

print("Dossiers prêts.")


Dossiers prêts.


## Partie 1 – Exploration du dataset

In [9]:
# Question 1 - Récupérer pour chaque image : nom, classe, format, mode,
# largeur, hauteur, ecart-type des pixels, nombre de canaux, taille (poids)

def explorer_image(chemin, classe):
    infos={
        "nom":os.path.basename(chemin),
        "classe":classe,
        "format":None,
        "mode":None,
        "largeur":None,
        "hauteur":None,
        "ecart_type":None,
        "nb_canaux":None,
        "taille_octets":os.path.getsize(chemin),
        "corrompue":False
    }
    try:
        with Image.open(chemin) as img:
            img.verify()  # Vérifie si l'image est corrompue
        with Image.open(chemin) as img:
            infos["format"]=img.format
            infos["mode"]=img.mode
            infos["largeur"], infos["hauteur"]=img.size
            if img.mode=="L":
                infos["nb_canaux"]=1
            else:
                infos["nb_canaux"]=len(img.getbands())
            stat=ImageStat.Stat(img.convert("L"))
            infos["ecart_type"]=stat.stddev[0]
    except Exception:
            infos["corrompue"]=True
    return infos


In [10]:
lignes = []
for classe in CLASSES:
    dossier_classe = os.path.join(RAW_DIR, classe)
    if not os.path.isdir(dossier_classe):
        continue
    for nom_fichier in os.listdir(dossier_classe):
        chemin = os.path.join(dossier_classe, nom_fichier)
        if os.path.isfile(chemin):
            lignes.append(explorer_image(chemin, classe))

df = pd.DataFrame(lignes)
print("Nombre total d'images explorées :", len(df))
df.head()

Nombre total d'images explorées : 1032


,nom,classe,format,mode,largeur,hauteur,ecart_type,nb_canaux,taille_octets,corrompue
0,cardboard1.jpg,cardboard,JPEG,RGB,512.0,384.0,31.875892,3.0,17333,False
1,cardboard10.jpg,cardboard,JPEG,RGB,512.0,384.0,38.799907,3.0,21683,False
2,cardboard100.jpg,cardboard,JPEG,RGB,512.0,384.0,44.498372,3.0,14884,False
3,cardboard101.jpg,cardboard,JPEG,RGB,512.0,384.0,68.561937,3.0,14289,False
4,cardboard102.jpg,cardboard,JPEG,RGB,512.0,384.0,45.320633,3.0,18015,False


## Partie 2 – Détecter les images corrompues

In [16]:
# Question 2 - Fonction de détection d'image corrompue
def est_corrompue(chemin):
    try:
        with Image.open(chemin) as img:
            img.verify()
        return False
    except Exception:
        return True

In [18]:
df_corrompues=df[df["corrompue"]==True]
print("Nombre d'images corrompues :", len(df_corrompues))
df_corrompues[["nom", "classe"]]

Nombre d'images corrompues : 6


,nom,classe
147,cardboard83.jpg,cardboard
326,glass74.jpg,glass
446,metal48.jpg,metal
633,paper213.jpg,paper
791,plastic13.jpg,plastic
1004,trash3.jpg,trash
